In [1]:
import sqlite3
from datetime import date

# Connect to database
conn = sqlite3.connect("attendance.db")
cursor = conn.cursor()
print("Connected to attendance.db")

# Create students table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL UNIQUE
)
""")

# Create attendance table
cursor.execute("""
CREATE TABLE IF NOT EXISTS attendance (
    attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    date TEXT NOT NULL,
    status TEXT NOT NULL CHECK(status IN ('Present', 'Absent')),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    UNIQUE (student_id, date)
)
""")

conn.commit()
print("Tables created successfully.")

Connected to attendance.db
Tables created successfully.


In [2]:
student_list = [
    ("Anna Reyes",),
    ("Carlos Dela Cruz",),
    ("Mika Santos",),
    ("Juan Garcia",),
    ("Maria Lopez",),
    ("Pedro Reyes",),
    ("Sofia Cruz",),
    ("Luis Torres",),
    ("Elena Ramos",),
    ("Marco Villanueva",)
]

cursor.executemany("""
INSERT OR IGNORE INTO students (full_name)
VALUES (?)
""", student_list)
conn.commit()

# Display all students
cursor.execute("SELECT * FROM students")
print("Registered Students:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

Registered Students:
  [1] Anna Reyes
  [2] Carlos Dela Cruz
  [3] Mika Santos
  [4] Juan Garcia
  [5] Maria Lopez
  [6] Pedro Reyes
  [7] Sofia Cruz
  [8] Luis Torres
  [9] Elena Ramos
  [10] Marco Villanueva


In [3]:
# Sample attendance records (student_id, date, status)
attendance_records = [
    (1, "2026-04-28", "Present"),
    (2, "2026-04-28", "Absent"),
    (3, "2026-04-28", "Present"),
    (4, "2026-04-28", "Present"),
    (5, "2026-04-28", "Absent"),
    (6, "2026-04-28", "Present"),
    (7, "2026-04-28", "Present"),
    (8, "2026-04-28", "Absent"),
    (9, "2026-04-28", "Present"),
    (10, "2026-04-28", "Present"),
    (1, "2026-04-29", "Present"),
    (2, "2026-04-29", "Present"),
    (3, "2026-04-29", "Absent"),
    (4, "2026-04-29", "Present"),
    (5, "2026-04-29", "Present"),
]

cursor.executemany("""
INSERT OR IGNORE INTO attendance (student_id, date, status)
VALUES (?, ?, ?)
""", attendance_records)
conn.commit()
print("Sample attendance records inserted.")

Sample attendance records inserted.


In [5]:
# Show all students
cursor.execute("SELECT * FROM students")
print("Students:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

# Get input
student_id = int(input("\nEnter Student ID: "))
date_input  = input("Enter date (YYYY-MM-DD) or press Enter for today: ")
status      = input("Enter status (Present / Absent): ").strip().capitalize()

# Use today's date if blank
if date_input.strip() == "":
    date_input = str(date.today())

# Validate status
if status not in ("Present", "Absent"):
    print("❌ Invalid status. Please enter 'Present' or 'Absent'.")
else:
    try:
        cursor.execute("""
        INSERT INTO attendance (student_id, date, status)
        VALUES (?, ?, ?)
        """, (student_id, date_input, status))
        conn.commit()
        print(f"✅ Attendance recorded: Student {student_id} → {status} on {date_input}")
    except sqlite3.IntegrityError:
        print(f"⚠️ Attendance for this student on {date_input} already exists!")

Students:
  [1] Anna Reyes
  [2] Carlos Dela Cruz
  [3] Mika Santos
  [4] Juan Garcia
  [5] Maria Lopez
  [6] Pedro Reyes
  [7] Sofia Cruz
  [8] Luis Torres
  [9] Elena Ramos
  [10] Marco Villanueva
⚠️ Attendance for this student on 2026-04-28 already exists!


In [6]:
# Show all students
cursor.execute("SELECT * FROM students")
print("Students:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

student_id = int(input("\nEnter Student ID to view attendance: "))

cursor.execute("""
SELECT s.full_name, a.date, a.status
FROM attendance a
JOIN students s ON a.student_id = s.student_id
WHERE a.student_id = ?
ORDER BY a.date ASC
""", (student_id,))

results = cursor.fetchall()
if results:
    print(f"\nAttendance record for {results[0][0]}:")
    print(f"  {'Date':<15} {'Status'}")
    print(f"  {'-'*25}")
    for row in results:
        print(f"  {row[1]:<15} {row[2]}")
else:
    print("No attendance records found for this student.")

Students:
  [1] Anna Reyes
  [2] Carlos Dela Cruz
  [3] Mika Santos
  [4] Juan Garcia
  [5] Maria Lopez
  [6] Pedro Reyes
  [7] Sofia Cruz
  [8] Luis Torres
  [9] Elena Ramos
  [10] Marco Villanueva

Attendance record for Anna Reyes:
  Date            Status
  -------------------------
  2026-04-28      Present
  2026-04-29      Present


In [7]:
date_input = input("Enter date to view attendance (YYYY-MM-DD): ")

cursor.execute("""
SELECT s.full_name, a.date, a.status
FROM attendance a
JOIN students s ON a.student_id = s.student_id
WHERE a.date = ?
ORDER BY s.full_name ASC
""", (date_input,))

results = cursor.fetchall()
if results:
    print(f"\nAttendance on {date_input}:")
    print(f"  {'Name':<25} {'Status'}")
    print(f"  {'-'*35}")
    for row in results:
        print(f"  {row[0]:<25} {row[2]}")
else:
    print(f"No attendance records found for {date_input}.")


Attendance on 2026-04-28:
  Name                      Status
  -----------------------------------
  Anna Reyes                Present
  Carlos Dela Cruz          Absent
  Elena Ramos               Present
  Juan Garcia               Present
  Luis Torres               Absent
  Marco Villanueva          Present
  Maria Lopez               Absent
  Mika Santos               Present
  Pedro Reyes               Present
  Sofia Cruz                Present


In [8]:
date_input = input("Enter date to check absences (YYYY-MM-DD): ")

cursor.execute("""
SELECT s.full_name, a.date, a.status
FROM attendance a
JOIN students s ON a.student_id = s.student_id
WHERE a.date = ? AND a.status = 'Absent'
ORDER BY s.full_name ASC
""", (date_input,))

results = cursor.fetchall()
if results:
    print(f"\nAbsent students on {date_input}:")
    for row in results:
        print(f"  ❌ {row[0]}")
else:
    print(f"No absences recorded on {date_input}.")


Absent students on 2026-04-28:
  ❌ Carlos Dela Cruz
  ❌ Luis Torres
  ❌ Maria Lopez


In [9]:
date_input = input("Enter date to check absences (YYYY-MM-DD): ")

cursor.execute("""
SELECT s.full_name, a.date, a.status
FROM attendance a
JOIN students s ON a.student_id = s.student_id
WHERE a.date = ? AND a.status = 'Absent'
ORDER BY s.full_name ASC
""", (date_input,))

results = cursor.fetchall()
if results:
    print(f"\nAbsent students on {date_input}:")
    for row in results:
        print(f"  ❌ {row[0]}")
else:
    print(f"No absences recorded on {date_input}.")


Absent students on 2026-04-28:
  ❌ Carlos Dela Cruz
  ❌ Luis Torres
  ❌ Maria Lopez


In [10]:
cursor.close()
conn.close()
print("Database connection closed.")

Database connection closed.
